# PhotoMappers Geolocation Visualization

**Author:** NUS PhotoMapper Team  
**Date:** 2026-09-24

## Objective
Build publication-quality visualizations for cross-view geolocalization performance using Recall@1, Recall@5, and Recall@10 metrics.

## Inputs
- `data/outputs/yearly_overall_performance.csv`
- `data/outputs/lifeline_overall_performance.csv`
- `data/outputs/community_lifelines_breakdown_by_year.csv`

## Outputs
- `data/outputs/Fig1_Yearly_Geolocation_Performance.png`
- `data/outputs/Fig2_Lifeline_Geolocation_Performance.png`
- `data/outputs/Fig3_Disaster_Type_Geolocation_Performance.png`

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
# Common visual grammar and Nature-style settings
METRIC_STYLE = {
    "R@1": {"label": "Recall@1", "color": "#08519C", "marker": "o", "lw": 2.4, "ms": 7.0, "zorder": 3},
    "R@5": {"label": "Recall@5", "color": "#D95F0E", "marker": "s", "lw": 2.0, "ms": 6.4, "zorder": 3},
    "R@10": {"label": "Recall@10", "color": "#238B45", "marker": "^", "lw": 2.0, "ms": 6.4, "zorder": 3},
}

CONNECTOR_COLOR = "#BDBDBD"
CONNECTOR_LW = 1.0
FIG_WIDTH = 7.0
FIGSIZE_YEARLY = (FIG_WIDTH, 4.2)
FIGSIZE_LIFELINE = (FIG_WIDTH, 5.0)
FIGSIZE_DISASTER = (FIG_WIDTH, 4.2)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 0.8,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.7,
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "savefig.facecolor": "none",
})

OUTPUT_DIR = Path.cwd().resolve() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TICKS_01 = np.linspace(0.0, 1.0, 6)

In [ ]:
# Build DataFrames directly from provided values
df_yearly = pd.DataFrame(
    [
        {"Year": 2017, "R@1": 0.300518135, "R@5": 0.440414508, "R@10": 0.518134715},
        {"Year": 2018, "R@1": 0.250000000, "R@5": 0.436567164, "R@10": 0.529850746},
        {"Year": 2019, "R@1": 0.298245614, "R@5": 0.491228070, "R@10": 0.596491228},
        {"Year": 2020, "R@1": 0.363636364, "R@5": 0.604895105, "R@10": 0.667832168},
        {"Year": 2021, "R@1": 0.443946188, "R@5": 0.632286996, "R@10": 0.744394619},
        {"Year": 2022, "R@1": 0.514970060, "R@5": 0.700598802, "R@10": 0.814371257},
        {"Year": 2023, "R@1": 0.650709220, "R@5": 0.751773050, "R@10": 0.822695035},
        {"Year": 2024, "R@1": 0.502645503, "R@5": 0.703703704, "R@10": 0.793650794},
        {"Year": 2025, "R@1": 0.494623656, "R@5": 0.709677419, "R@10": 0.752688172},
    ]
)

df_lifeline = pd.DataFrame(
    [
        {"Lifeline": "Hazardous Materials", "R@1": 0.500000000, "R@5": 0.500000000, "R@10": 0.500000000},
        {"Lifeline": "Water Systems", "R@1": 0.500000000, "R@5": 1.000000000, "R@10": 1.000000000},
        {"Lifeline": "Communications", "R@1": 0.000000000, "R@5": 0.600000000, "R@10": 0.600000000},
        {"Lifeline": "Health and Medical", "R@1": 0.500000000, "R@5": 0.687500000, "R@10": 0.687500000},
        {"Lifeline": "Energy", "R@1": 0.436893204, "R@5": 0.650485437, "R@10": 0.737864078},
        {"Lifeline": "Safety and Security", "R@1": 0.452054795, "R@5": 0.643835616, "R@10": 0.719178082},
        {"Lifeline": "Food Hydration Shelter", "R@1": 0.400000000, "R@5": 0.586666667, "R@10": 0.685333333},
        {"Lifeline": "Transportation", "R@1": 0.512520868, "R@5": 0.752921536, "R@10": 0.813021703},
        {"Lifeline": "Other", "R@1": 0.466666667, "R@5": 0.676190476, "R@10": 0.785714286},
    ]
)

df_disaster = pd.DataFrame(
    [
        {"Disaster Type": "Wildfire", "R@1": 0.818181818, "R@5": 0.818181818, "R@10": 1.000000000},
        {"Disaster Type": "Earthquake", "R@1": 0.350000000, "R@5": 0.583333333, "R@10": 0.650000000},
        {"Disaster Type": "Flood", "R@1": 0.562962963, "R@5": 0.814814815, "R@10": 0.851851852},
        {"Disaster Type": "Tornado", "R@1": 0.353535354, "R@5": 0.555555556, "R@10": 0.641414141},
        {"Disaster Type": "Tropical Cyclone", "R@1": 0.452547453, "R@5": 0.668331668, "R@10": 0.756243756},
        {"Disaster Type": "Other/Unknown", "R@1": 0.351380042, "R@5": 0.531847134, "R@10": 0.616772824},
    ]
)

# Harmonize a display label for requested comma-separated category text
df_lifeline["Lifeline Display"] = df_lifeline["Lifeline"].replace({
    "Food Hydration Shelter": "Food, Hydration, Shelter"
})

In [ ]:
def apply_nature_axes(ax, add_horizontal_grid=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    if add_horizontal_grid:
        ax.grid(axis="y", linestyle="-")
    else:
        ax.grid(False)


def plot_yearly(ax, df):
    for metric in ["R@1", "R@5", "R@10"]:
        style = METRIC_STYLE[metric]
        ax.plot(
            df["Year"],
            df[metric],
            color=style["color"],
            marker=style["marker"],
            lw=style["lw"],
            ms=style["ms"],
            label=style["label"],
            zorder=style["zorder"],
        )

    ax.set_xlim(2017, 2025)
    ax.set_xticks(df["Year"].tolist())
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks(TICKS_01)
    ax.set_xlabel("Year")
    ax.set_ylabel("Recall")
    apply_nature_axes(ax, add_horizontal_grid=True)


def plot_dot_range(ax, df, category_col, order, other_label=None):
    plot_df = df.copy()
    if other_label is not None and other_label in plot_df[category_col].values:
        core_order = [x for x in order if x != other_label]
        y_map = {name: idx + 1 for idx, name in enumerate(core_order)}
        y_map[other_label] = 0
    else:
        y_map = {name: idx for idx, name in enumerate(order)}

    plot_df = plot_df[plot_df[category_col].isin(y_map)].copy()
    plot_df["_y"] = plot_df[category_col].map(y_map)

    for _, row in plot_df.iterrows():
        ax.hlines(
            y=row["_y"],
            xmin=row["R@1"],
            xmax=row["R@10"],
            color=CONNECTOR_COLOR,
            lw=CONNECTOR_LW,
            zorder=1,
        )

    for metric in ["R@1", "R@5", "R@10"]:
        style = METRIC_STYLE[metric]
        ax.scatter(
            plot_df[metric],
            plot_df["_y"],
            color=style["color"],
            marker=style["marker"],
            s=(style["ms"] ** 2) * 1.9,
            label=style["label"],
            zorder=3,
        )

    sorted_items = sorted(y_map.items(), key=lambda x: x[1])
    ax.set_yticks([v for _, v in sorted_items])
    ax.set_yticklabels([k for k, _ in sorted_items])
    ax.set_xlim(0.0, 1.0)
    ax.set_xticks(TICKS_01)
    ax.set_xlabel("Recall")
    apply_nature_axes(ax, add_horizontal_grid=False)
    ax.grid(axis="x", visible=False)


def add_compact_legend(ax, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.02)):
    handles, labels = ax.get_legend_handles_labels()
    seen = {}
    unique_handles, unique_labels = [], []
    for h, l in zip(handles, labels):
        if l not in seen:
            seen[l] = True
            unique_handles.append(h)
            unique_labels.append(l)
    ax.legend(
        unique_handles,
        unique_labels,
        ncol=ncol,
        frameon=False,
        loc=loc,
        bbox_to_anchor=bbox_to_anchor,
        handletextpad=0.4,
        columnspacing=1.1,
        borderaxespad=0.1,
    )

In [ ]:
# Data validation
def validate_dataframe(df, name, category_col=None, expected_categories=None):
    metric_cols = ["R@1", "R@5", "R@10"]
    vals = df[metric_cols]

    in_range = ((vals >= 0.0) & (vals <= 1.0)).all().all()
    monotonic = ((df["R@1"] <= df["R@5"]) & (df["R@5"] <= df["R@10"])).all()
    missing_total = int(df.isna().sum().sum())

    category_ok = True
    missing_categories = []
    extra_categories = []
    if category_col and expected_categories is not None:
        observed = set(df[category_col].tolist())
        expected = set(expected_categories)
        missing_categories = sorted(expected - observed)
        extra_categories = sorted(observed - expected)
        category_ok = (len(missing_categories) == 0 and len(extra_categories) == 0)

    return {
        "dataset": name,
        "in_range_0_1": bool(in_range),
        "monotonic_R1_R5_R10": bool(monotonic),
        "missing_values": missing_total,
        "category_ok": bool(category_ok),
        "missing_categories": missing_categories,
        "extra_categories": extra_categories,
    }


expected_years = list(range(2017, 2026))
expected_lifelines = [
    "Safety and Security",
    "Food, Hydration, Shelter",
    "Health and Medical",
    "Energy",
    "Communications",
    "Transportation",
    "Hazardous Materials",
    "Water Systems",
    "Other",
]
expected_disaster_types = [
    "Tropical Cyclone",
    "Flood",
    "Tornado",
    "Earthquake",
    "Wildfire",
    "Other/Unknown",
]

year_ok = sorted(df_yearly["Year"].tolist()) == expected_years

summary = []
summary.append(validate_dataframe(df_yearly, "df_yearly"))
summary.append(
    validate_dataframe(
        df_lifeline.rename(columns={"Lifeline Display": "Lifeline Expected"}),
        "df_lifeline",
        category_col="Lifeline Expected",
        expected_categories=expected_lifelines,
    )
)
summary.append(
    validate_dataframe(
        df_disaster,
        "df_disaster",
        category_col="Disaster Type",
        expected_categories=expected_disaster_types,
    )
)

print("Validation Summary")
print("-" * 72)
print(f"Years 2017-2025 fully represented: {year_ok}")
for item in summary:
    print(
        f"{item['dataset']}: in_range_0_1={item['in_range_0_1']}, "
        f"monotonic={item['monotonic_R1_R5_R10']}, "
        f"missing_values={item['missing_values']}, "
        f"categories_ok={item['category_ok']}"
    )
    if item["missing_categories"]:
        print(f"  Missing categories: {item['missing_categories']}")
    if item["extra_categories"]:
        print(f"  Extra categories: {item['extra_categories']}")

In [ ]:
# Figure 1: Yearly performance
fig1, ax1 = plt.subplots(figsize=FIGSIZE_YEARLY)
plot_yearly(ax1, df_yearly)
add_compact_legend(ax1, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.02))
fig1_path = OUTPUT_DIR / "Fig1_Yearly_Geolocation_Performance.png"
fig1.savefig(fig1_path, dpi=600, transparent=True, bbox_inches="tight")
plt.show()
plt.close(fig1)

In [ ]:
# Figure 2: Community Lifelines
lifeline_order = [
    "Safety and Security",
    "Food, Hydration, Shelter",
    "Health and Medical",
    "Energy",
    "Communications",
    "Transportation",
    "Hazardous Materials",
    "Water Systems",
    "Other",
]

df_lifeline_plot = df_lifeline.copy()
df_lifeline_plot["Lifeline"] = df_lifeline_plot["Lifeline Display"]

fig2, ax2 = plt.subplots(figsize=FIGSIZE_LIFELINE)
plot_dot_range(ax2, df_lifeline_plot, "Lifeline", lifeline_order, other_label="Other")
add_compact_legend(ax2, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.04))
fig2_path = OUTPUT_DIR / "Fig2_Lifeline_Geolocation_Performance.png"
fig2.savefig(fig2_path, dpi=600, transparent=True, bbox_inches="tight")
plt.show()
plt.close(fig2)

In [ ]:
# Figure 3: Disaster types
disaster_order = [
    "Tropical Cyclone",
    "Flood",
    "Tornado",
    "Earthquake",
    "Wildfire",
    "Other/Unknown",
]

fig3, ax3 = plt.subplots(figsize=FIGSIZE_DISASTER)
plot_dot_range(ax3, df_disaster, "Disaster Type", disaster_order, other_label=None)
add_compact_legend(ax3, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.04))
fig3_path = OUTPUT_DIR / "Fig3_Disaster_Type_Geolocation_Performance.png"
fig3.savefig(fig3_path, dpi=600, transparent=True, bbox_inches="tight")
plt.show()
plt.close(fig3)

In [ ]:
# Figure 4: One-column overview with shared legend
fig4, axes = plt.subplots(
    3,
    1,
    figsize=(FIG_WIDTH, 12.0),
    gridspec_kw={"height_ratios": [1.0, 1.1, 1.0]},
)

ax_a, ax_b, ax_c = axes

plot_yearly(ax_a, df_yearly)
ax_a.text(0.0, 1.03, "a", transform=ax_a.transAxes, fontsize=13, fontweight="bold")

plot_dot_range(ax_b, df_lifeline_plot, "Lifeline", lifeline_order, other_label="Other")
ax_b.set_xlabel("Recall")
ax_b.text(0.0, 1.03, "b", transform=ax_b.transAxes, fontsize=13, fontweight="bold")

plot_dot_range(ax_c, df_disaster, "Disaster Type", disaster_order, other_label=None)
ax_c.set_xlabel("Recall")
ax_c.text(0.0, 1.03, "c", transform=ax_c.transAxes, fontsize=13, fontweight="bold")

# Shared legend
shared_handles = []
shared_labels = []
seen = set()
for axis in [ax_a, ax_b, ax_c]:
    h, l = axis.get_legend_handles_labels()
    for hi, li in zip(h, l):
        if li not in seen:
            seen.add(li)
            shared_handles.append(hi)
            shared_labels.append(li)

fig4.legend(
    shared_handles,
    shared_labels,
    ncol=3,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    handletextpad=0.4,
    columnspacing=1.1,
)

fig4.tight_layout(rect=[0.0, 0.0, 1.0, 0.97], h_pad=2.0)
fig4_path = OUTPUT_DIR / "Fig4_Geolocation_Overview.png"
fig4.savefig(fig4_path, dpi=600, transparent=True, bbox_inches="tight")
plt.show()
plt.close(fig4)

In [ ]:
# Additional descriptive summaries for interpretation
yearly_gain = df_yearly.assign(
    gain_R5_R1=df_yearly["R@5"] - df_yearly["R@1"],
    gain_R10_R1=df_yearly["R@10"] - df_yearly["R@1"],
)

lifeline_gain = df_lifeline_plot.assign(
    gain_R5_R1=df_lifeline_plot["R@5"] - df_lifeline_plot["R@1"],
    gain_R10_R1=df_lifeline_plot["R@10"] - df_lifeline_plot["R@1"],
)

disaster_gain = df_disaster.assign(
    gain_R5_R1=df_disaster["R@5"] - df_disaster["R@1"],
    gain_R10_R1=df_disaster["R@10"] - df_disaster["R@1"],
)

print("Mean retrieval gains:")
print(f"  Yearly   : R@5-R@1={yearly_gain['gain_R5_R1'].mean():.3f}, R@10-R@1={yearly_gain['gain_R10_R1'].mean():.3f}")
print(f"  Lifeline : R@5-R@1={lifeline_gain['gain_R5_R1'].mean():.3f}, R@10-R@1={lifeline_gain['gain_R10_R1'].mean():.3f}")
print(f"  Disaster : R@5-R@1={disaster_gain['gain_R5_R1'].mean():.3f}, R@10-R@1={disaster_gain['gain_R10_R1'].mean():.3f}")

## Concise Scientific Interpretation

Across years, geolocalization recall generally improves from 2017 to 2023, with a visible softening in 2024-2025 relative to the 2023 peak. This indicates temporal variation rather than a strictly monotonic trend.

Across Community Lifelines, performance differs by category, and the recall spread between Recall@1 and Recall@10 varies substantially. Several categories show moderate-to-large retrieval gains (Recall@5 - Recall@1 and Recall@10 - Recall@1), highlighting that broader candidate retrieval improves outcomes beyond top-1 matching.

Across Disaster Types, recall profiles also vary, with some categories showing stronger top-1 performance and others showing larger gains when moving to top-5 or top-10 retrieval. This suggests heterogeneity in retrieval behavior across event types at different cutoff levels.

In all three analyses, Recall@5 and Recall@10 are consistently higher than Recall@1, confirming non-trivial gains from considering additional retrieved candidates. Interpret extreme values cautiously (for example Water Systems and Wildfire) because sample sizes are not provided here and may be small.

These plots are descriptive and do not support causal conclusions about why differences occur.

In [ ]:
exported = [
    fig1_path,
    fig2_path,
    fig3_path,
    fig4_path,
]

print("Exported figure files:")
for p in exported:
    print(f"- {p.as_posix()}")